# Session 9 — Parsing (Constituency, CFG, Chart & Probabilistic CKY)

> Interactive study notes for **NLP Session 9 – Parsing** (Dr. Chetana Gavankar).
> Everything here is **pure Python, no installs** — just run each cell top to bottom.
> There's an optional NLTK demo at the very end.

**What "parsing" means in one line:** taking a flat sentence of words and discovering
its hidden **structure** — which words group together into phrases, and how those
phrases nest to form the whole sentence.

Two ingredients (from the slides):

| Ingredient | What it is |
|------------|-----------|
| **Grammar** | a *formal specification* of the structures allowed in the language |
| **Parsing technique** | a *method* of analysing a sentence to find its structure per the grammar |

### Roadmap of this notebook
1. Why parse at all? (applications)
2. Constituency vs. Dependency — two views of structure
3. Context-Free Grammars (CFG) + phrase categories
4. Top-down vs. Bottom-up parsing (see them run)
5. Chart parsing — why we cache partial results (the "dot" idea)
6. **CKY** recogniser from scratch
7. **PCFG** — adding probabilities; probability of a tree
8. **Probabilistic CKY** — the *"The flight includes a meal"* slide example, coded
9. Chomsky Normal Form (why CKY needs it)
10. Problems with PCFGs
11. Evaluating a parser — precision / recall / F1


## 1 — Why parse? (Applications)

Parsing is not academic decoration — the *meaning* often lives in the structure:

- **Sentiment**: *"I like Frozen"* vs *"I do **not** like Frozen"* — almost the same words,
  opposite meaning. Structure (where the *not* attaches) decides it.
- **Relation extraction**: *"Rome is the capital of Italy and the region of Lazio"* —
  entities alone aren't enough; parsing tells us *what is the capital of what*.
- **Question answering**: parse the question, then match it against parsed sentences.
- **Machine translation / grammar checking / speech recognition**: parsing scores
  whether a word-string is a *plausible* sentence (a language model).

**Ambiguity is explosive** — the classic headache. One sentence can have *many* legal
parses, and the count blows up with length. That's exactly why we later add
**probabilities** (PCFGs) to rank them.


In [ ]:
# A famous structurally-ambiguous sentence: "I saw the man with the telescope."
# Who has the telescope? Two readings => two different tree structures.
readings = {
    "Reading A — I used the telescope to see the man":
        "[S [NP I] [VP [V saw] [NP the man] [PP with the telescope]]]",
    "Reading B — the man who had the telescope":
        "[S [NP I] [VP [V saw] [NP the man [PP with the telescope]]]]",
}
for meaning, bracketing in readings.items():
    print(meaning)
    print("   ", bracketing, "\n")
print("Same 7 words -> two structures. Parsing must choose; PCFGs choose by probability.")


## 2 — Two views of linguistic structure

The slides contrast two ways to describe sentence structure:

**1. Constituency (phrase structure)** — words nest into *constituents* (NP, VP, …).
A constituent is a group that *behaves as a unit*: it can move around together.

> *John talked **[to the children] [about drugs]**.*
> *John talked **[about drugs] [to the children]**.*  ✅ (the bracketed chunks move as units)
> *\*John talked drugs to the children about.*  ❌

**2. Dependency structure** — skip the phrases; just draw arrows showing which word
*depends on* (modifies / is an argument of) which. That's **Session 10's** topic
([`../dependency-parsing/arc_eager_parser.ipynb`](../dependency-parsing/arc_eager_parser.ipynb)).

**Session 9 = the constituency / phrase-structure view.** Everything below builds
phrase trees.


## 3 — Context-Free Grammar (CFG)

A **CFG** is a set of rewrite rules. Each rule has **one symbol on the left** (the
*mother*) that expands into a sequence of symbols on the right. "Context-free" = the
left side is a single symbol, so you can apply the rule regardless of surrounding context.

**Phrase categories (heads):**

| Phrase | Head word | Typically starts with |
|--------|-----------|-----------------------|
| **NP** (noun phrase) | noun | article or noun |
| **VP** (verb phrase) | verb | a verb |
| **ADJP** (adjective phrase) | adjective | an adjective |
| **ADVP** (adverb phrase) | adverb | an adverb |
| **PP** (prepositional phrase) | preposition | a preposition |

Two kinds of rules:
- **Grammar rules** — non-terminal → sequence, e.g. `S → NP VP`
- **Lexical rules** — non-terminal → actual word, e.g. `N → NLP`, `V → is`

Let's encode the tiny grammar straight from **slide 10** and check it accepts
*"NLP is very interesting"*.


In [ ]:
# --- Tiny CFG from slide 10 -------------------------------------------------
# S  -> NP VP
# VP -> V NP
# NP -> N | Adj NP | Adj
# N -> NLP ;  V -> is ;  Adj -> very | interesting
grammar_rules = [
    ("S",  ["NP", "VP"]),
    ("VP", ["V", "NP"]),
    ("NP", ["N"]),
    ("NP", ["Adj", "NP"]),
    ("NP", ["Adj"]),
]
lexicon = {
    "NLP": ["N"],
    "is": ["V"],
    "very": ["Adj"],
    "interesting": ["Adj"],
}

sentence = "NLP is very interesting".split()
print("Sentence:", sentence)
print("Each word's possible categories:")
for w in sentence:
    print(f"   {w:>12} -> {lexicon.get(w, ['?'])}")


## 4 — Top-down vs. Bottom-up parsing

Two opposite search directions for finding a tree:

| | **Top-down** | **Bottom-up** |
|-|--------------|----------------|
| Starts from | the **start symbol `S`** (the root) | the **words** (the leaves) |
| Builds tree | root → leaves | leaves → root |
| Basic move | expand a non-terminal using a rule's RHS | match a span of symbols to a rule's RHS, replace with its LHS |
| Weakness | may predict structures the words can't support | may build constituents that go nowhere |

**Bottom-up**, in the slides' words: start with the word list; repeatedly
(a) rewrite a word by a lexical category, or (b) replace a symbol sequence that
matches some rule's RHS by that rule's LHS — until you reach `S` spanning everything.

Below is a small **bottom-up recogniser** that reduces the sentence step by step and
prints its reasoning so you can *watch* it climb from words to `S`.


In [ ]:
def bottom_up_trace(words, grammar_rules, lexicon, max_steps=50):
    """Greedy bottom-up recogniser -- prints each reduction it makes.
    (Greedy = for teaching; real parsers search all options / use a chart.)"""
    # Step 1: rewrite each word as one lexical category (first listed).
    stack = [lexicon[w][0] for w in words]
    print("start :", stack, "   (words rewritten to categories)")
    for step in range(max_steps):
        reduced = False
        # try to match the RHS of some rule against a contiguous slice of the stack
        for lhs, rhs in grammar_rules:
            n = len(rhs)
            for i in range(len(stack) - n + 1):
                if stack[i:i + n] == rhs:
                    new = stack[:i] + [lhs] + stack[i + n:]
                    print(f"reduce: {rhs} -> {lhs:<3} giving {new}")
                    stack = new
                    reduced = True
                    break
            if reduced:
                break
        if not reduced:
            break
    ok = (stack == ["S"])
    print("result:", stack, "=>", "ACCEPTED (is a sentence)" if ok else "not reduced to S")
    return ok

bottom_up_trace(sentence, grammar_rules, lexicon)


> The greedy version can get stuck when a word has several possible categories or a
> span can reduce in several ways (ambiguity again!). The fix used in practice is a
> **chart** that stores *every* partial result so no work — and no option — is lost.


## 5 — Chart parsing (caching partial results)

Naive parsers redo the same matches again and again. A **chart** is a data structure
that **stores partial results** (constituents found, and rules partly matched) so work
is never repeated.

**The "dot" notation (slides 23–24).** As we read a rule left to right we mark how far
we've got with a dot `•`:

```
Suppose input starts with ART. Rules starting with ART get activated:
   NP → ART • ADJ N        (seen ART, still expecting ADJ N)
   NP → ART • N            (seen ART, still expecting N)
Next input is ADJ, so extend the first one:
   NP → ART ADJ • N        (now only expecting N)
```

- A rule with the dot **not** at the end = an **active arc** (partial, incomplete).
- A rule with the dot **at the end** = a **completed constituent**.

**Why the chart wins (slide 33):**
- ✅ No repeated computation of the same sub-problem
- ✅ Handles **left-recursive** grammars
- ✅ Handles **ambiguity** (keeps all analyses)
- ✅ **No backtracking** needed

The cleanest famous chart algorithm for the bottom-up case is **CKY**, next.


## 6 — CKY recogniser from scratch

**CKY** (Cocke–Kasami–Younger) is the classic **bottom-up dynamic-programming** chart
parser. It fills a triangular table where cell `[i][j]` holds every non-terminal that
can span words `i … j`.

**Requirement:** the grammar must be in **Chomsky Normal Form (CNF)** — every rule is
either `A → B C` (two non-terminals) or `A → w` (one word). (We cover *why*, and how to
convert, in section 9.)

**The idea:** a phrase covering a span is built from **two smaller phrases** that meet at
some **split point** `k`. So to fill cell `[i][j]`, try every split `k` between `i` and
`j`, and every rule `A → B C` where `B` sits in `[i][k]` and `C` sits in `[k][j]`.

We'll use a CNF grammar for *"the man saw the dog"*.


In [ ]:
from collections import defaultdict

# --- A small CNF grammar ----------------------------------------------------
# binary rules:  A -> B C
binary = [
    ("S",  "NP", "VP"),
    ("VP", "V",  "NP"),
    ("NP", "Det", "N"),
]
# lexical rules: A -> word
lexical = {
    "the": ["Det"],
    "man": ["N"],
    "dog": ["N"],
    "saw": ["V"],
}

def cky_recognise(words, binary, lexical, verbose=True):
    n = len(words)
    # table[i][j] = set of non-terminals spanning words[i:j]  (j exclusive)
    table = [[set() for _ in range(n + 1)] for _ in range(n + 1)]
    # 1) lexical / diagonal fill: spans of length 1
    for i, w in enumerate(words):
        for cat in lexical.get(w, []):
            table[i][i + 1].add(cat)
    # 2) fill longer spans, shortest first
    for span in range(2, n + 1):            # span length
        for i in range(0, n - span + 1):    # start
            j = i + span                    # end (exclusive)
            for k in range(i + 1, j):       # split point
                left, right = table[i][k], table[k][j]
                for (A, B, C) in binary:
                    if B in left and C in right:
                        table[i][j].add(A)
    if verbose:
        for span in range(1, n + 1):
            for i in range(0, n - span + 1):
                j = i + span
                if table[i][j]:
                    print(f"span [{i}:{j}] {' '.join(words[i:j]):<20} -> {sorted(table[i][j])}")
    accepted = "S" in table[0][n]
    print("\
Accepted as a sentence?", accepted, "  (is 'S' in the top cell?)")
    return table, accepted

words2 = "the man saw the dog".split()
_ = cky_recognise(words2, binary, lexical)


**Try it yourself:** change `words2` to `"the dog saw the man"` (works) or
`"man the saw dog"` (rejected — `S` never appears in the top cell). The chart shows you
*exactly* which sub-spans did and didn't form constituents.


## 7 — PCFG: adding probabilities

A **Probabilistic CFG** augments a CFG by attaching a **probability to every rule**.

**Formal definition (slide 35).** A PCFG = terminals + non-terminals + a start symbol +
rules `N_i → ξ_j` + a probability for each rule, such that for every non-terminal the
probabilities of its rules **sum to 1**:

$$\forall i \quad \sum_j P(N_i \to \xi_j) = 1$$

Read `P(A → β)` as: *given that we're expanding A, how likely is this particular
expansion?* — a conditional probability `P(A → β | A)`. A grammar where every LHS sums
to 1 is called **consistent**.

**Probability of a parse tree (slide 36):** multiply the probabilities of *all* the
rules used to build it:

$$P(T) = \prod_{i} P(\text{rule}_i)$$

**Probability of a sentence:** sum over all trees that yield it:

$$P(w) = \sum_{t:\, \text{yield}(t)=w} P(t)$$

The **best parse** is the tree with the highest probability — `argmax_t P(t)`. That's how
PCFGs resolve ambiguity (e.g. the telescope sentence, PP-attachment).


In [ ]:
# A PCFG as {LHS: [(rhs_tuple, prob), ...]}. Probs for each LHS sum to 1.
pcfg = {
    "S":  [(("NP", "VP"), 1.0)],
    "VP": [(("V", "NP"), 0.7), (("VP", "PP"), 0.3)],
    "NP": [(("NP", "PP"), 0.4), (("N",), 0.6)],
    "PP": [(("P", "NP"), 1.0)],
    "V":  [(("saw",), 1.0)],
    "P":  [(("with",), 1.0)],
    "N":  [(("astronomers",), 0.4), (("stars",), 0.3), (("ears",), 0.3)],
}

# sanity check: does every LHS sum to 1?
print("Consistency check (each LHS should sum to 1.0):")
for lhs, prods in pcfg.items():
    total = round(sum(p for _, p in prods), 6)
    print(f"   {lhs:>3}: {total}  {'OK' if abs(total-1) < 1e-9 else 'NOT CONSISTENT!'}")

def tree_prob(rules_used):
    """Product of the probabilities of every rule used to build a tree."""
    prob = 1.0
    for p in rules_used:
        prob *= p
    return prob

# Probability of ONE concrete tree = product of every rule used to build it.
# Parse of "astronomers saw stars":
#   S -> NP VP           (1.0)
#   NP -> N (0.6), N -> astronomers (0.4)
#   VP -> V NP           (0.7)
#     V -> saw (1.0),  NP -> N (0.6), N -> stars (0.3)
rules_used = [1.0,           # S  -> NP VP
              0.6, 0.4,      # NP -> N ; N -> astronomers
              0.7,           # VP -> V NP
              1.0, 0.6, 0.3] # V -> saw ; NP -> N ; N -> stars
print(f"\nP(tree for 'astronomers saw stars') = {tree_prob(rules_used):.6g}")
print("That's the P(T) = product of rule probabilities formula in action.")
print("When a sentence has SEVERAL trees, we pick argmax P(T) -- done for real")
print("with back-pointers in the Probabilistic CKY section below.")


## 8 — Probabilistic CKY: the slide's worked example, in code

Now the headline algorithm. **Probabilistic CKY** is plain CKY where each cell stores a
**probability** (best way to build that non-terminal over that span) plus a
**back-pointer** (which rule + split produced it) so we can rebuild the winning tree.

We reproduce **slides 66–76** exactly — grammar and sentence *"The flight includes a meal"*:

```
S  → NP VP  [.80]      V   → includes [.05]
NP → Det N  [.30]      Det → the      [.40]
VP → V  NP  [.20]      Det → a        [.40]
                       N   → flight   [.02]
                       N   → meal     [.01]
```

When we combine `B` in `[i][k]` with `C` in `[k][j]` via rule `A → B C [p]`, the new
probability is:

$$P(A,\,i,j) = p \times P(B,\,i,k) \times P(C,\,k,j)$$

Keeping the **max** over all splits gives the most probable parse (Viterbi-style).


In [ ]:
# --- PCFG from slides 66-76 -------------------------------------------------
P_binary = [   # (A -> B C, prob)
    ("S",  "NP", "VP", 0.80),
    ("NP", "Det", "N", 0.30),
    ("VP", "V",  "NP", 0.20),
]
P_lexical = {  # word -> list of (tag, prob)
    "the":      [("Det", 0.40)],
    "a":        [("Det", 0.40)],
    "includes": [("V",   0.05)],
    "flight":   [("N",   0.02)],
    "meal":     [("N",   0.01)],
}

def prob_cky(words, P_binary, P_lexical):
    n = len(words)
    # cell[i][j][A] = (best_prob, backpointer)
    cell = [[dict() for _ in range(n + 1)] for _ in range(n + 1)]

    # lexical step (spans of length 1)
    for i, w in enumerate(words):
        for tag, p in P_lexical.get(w, []):
            cell[i][i + 1][tag] = (p, ("leaf", w))

    # syntactic step (longer spans)
    for span in range(2, n + 1):
        for i in range(0, n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                left, right = cell[i][k], cell[k][j]
                for (A, B, C, p) in P_binary:
                    if B in left and C in right:
                        prob = p * left[B][0] * right[C][0]
                        if A not in cell[i][j] or prob > cell[i][j][A][0]:
                            cell[i][j][A] = (prob, ("bin", A, B, C, i, k, j))
    return cell

def build_tree(cell, i, j, A):
    """Follow back-pointers to reconstruct the winning tree as nested lists."""
    prob, bp = cell[i][j][A]
    if bp[0] == "leaf":
        return [A, bp[1]]
    _, A, B, C, i, k, j = bp
    return [A, build_tree(cell, i, k, B), build_tree(cell, k, j, C)]

def show_tree(t, indent=0):
    if len(t) == 2 and isinstance(t[1], str):     # leaf: [tag, word]
        print("  " * indent + f"{t[0]} -> '{t[1]}'")
    else:
        print("  " * indent + t[0])
        for child in t[1:]:
            show_tree(child, indent + 1)

sent = "the flight includes a meal".split()
cell = prob_cky(sent, P_binary, P_lexical)

top = cell[0][len(sent)]
print("Top-cell result:", {k: round(v[0], 12) for k, v in top.items()})
print(f"\nP(best parse of '{' '.join(sent)}') = {top['S'][0]:.3e}\n")
print("Most probable parse tree:")
show_tree(build_tree(cell, 0, len(sent), "S"))


**Check against the slide:** slide 76 computes
`.8 × .3 × .4 × .02 × .2 × .05 × .3 × .4 × .01`. Our code prints the same
`P(best parse) ≈ 2.304e-08`. That single number is the product of *every* rule
probability in the tree — exactly the `P(T) = ∏ P(rule)` formula from section 7.

**Summary of prob-CKY (slide 77):** cells hold probabilities; a bottom-up pass computes
the best parse incrementally; back-pointers let us traverse the table *backwards* to read
off the tree.


## 9 — Chomsky Normal Form (why CKY needs it)

CKY only works on grammars in **Chomsky Normal Form (CNF)**: every rule is either
`A → B C` (two non-terminals) or `A → w` (one terminal). That uniform shape is what lets
CKY "split every span into exactly two parts."

**Good news:** *any* CFG can be rewritten in CNF **without changing the language** it
accepts. Three fix-ups (slides 62–65):

| Problem rule | Why it's illegal | Fix |
|--------------|------------------|-----|
| `NP → the Nominal` | mixes a terminal + non-terminal | add dummy `Det → the`, rewrite `NP → Det Nominal` |
| `NP → Nominal` (unit production) | single non-terminal on RHS | inline `Nominal`'s rules into `NP` |
| `NP → Det Noun PP` (>2 symbols) | more than two on RHS | introduce a new symbol: `NP → Det X`, `X → Noun PP` |

The function below mechanises the third (and most common) fix — **binarising** long
rules — which is the step CKY really depends on.


In [ ]:
def binarise(rules):
    """Turn every rule A -> X1 X2 ... Xn (n > 2) into a chain of binary rules.
       Returns a new list containing only unary/binary rules."""
    out, counter = [], [0]
    def new_symbol(base):
        counter[0] += 1
        return f"{base}@{counter[0]}"
    for lhs, rhs in rules:
        rhs = list(rhs)
        while len(rhs) > 2:
            # take the LAST two, fold them into a fresh symbol
            x = new_symbol(lhs)
            out.append((x, [rhs[-2], rhs[-1]]))
            rhs = rhs[:-2] + [x]
        out.append((lhs, rhs))
    return out

long_rules = [
    ("NP", ["Det", "Adj", "N", "PP"]),   # 4 symbols on the RHS
    ("S",  ["NP", "VP"]),                 # already binary -> untouched
]
print("Before:")
for r in long_rules: print("   ", r[0], "->", " ".join(r[1]))
print("\
After binarising (CKY-ready):")
for lhs, rhs in binarise(long_rules):
    print("   ", lhs, "->", " ".join(rhs))


## 10 — Problems with PCFGs

PCFGs are a great baseline but have real blind spots (slides 38, 78). Their probabilities
depend on **structure only, not on the actual words**:

- **No lexicalisation.** `VP → V NP NP` should be far likelier for *hand* / *tell*
  (*"hand me the book"*) than for *sleep*. A plain PCFG can't see the verb, so it can't
  tell. → fix: **lexicalised PCFGs**.
- **No context / structural context.** How an `NP` expands often depends on its
  *position* (subject NPs are more often pronouns than object NPs). A PCFG treats every
  `NP` identically.
- Probabilities are **structural, not semantic** — so a PCFG alone is a weak model, but
  it **combines well with an n-gram / trigram model**.

On the plus side: PCFGs give a real probabilistic **language model**, are **robust**, and
are good for **grammar induction**.


## 11 — Evaluating a parser (precision / recall / F1)

How good is a parser? Compare the **constituents (brackets)** it produces against a
**gold** hand-annotated tree. Each constituent is a triple `(label, start, end)`.

- **Precision** = of the brackets the parser produced, what fraction are correct?
- **Recall** = of the gold brackets, what fraction did the parser find?
- **F1** = harmonic mean of the two (the standard single-number score — *PARSEVAL*).

$$\text{P}=\frac{|\text{correct}|}{|\text{produced}|}\quad
\text{R}=\frac{|\text{correct}|}{|\text{gold}|}\quad
\text{F1}=\frac{2PR}{P+R}$$


In [ ]:
def eval_parse(gold, predicted):
    """gold, predicted: sets of (label, i, j) constituent brackets."""
    gold, predicted = set(gold), set(predicted)
    correct = gold & predicted
    precision = len(correct) / len(predicted) if predicted else 0.0
    recall    = len(correct) / len(gold)      if gold      else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

# Toy example: parser gets the VP span slightly wrong and misses one NP.
gold = {("S",0,5), ("NP",0,2), ("VP",2,5), ("V",2,3), ("NP",3,5), ("Det",3,4), ("N",4,5)}
pred = {("S",0,5), ("NP",0,2), ("VP",2,5), ("V",2,3), ("Det",3,4), ("N",4,5), ("NP",3,4)}

p, r, f = eval_parse(gold, pred)
print(f"Precision = {p:.2%}")
print(f"Recall    = {r:.2%}")
print(f"F1        = {f:.2%}")
print("\
Missed (in gold, not predicted):", gold - pred)
print("Spurious (predicted, not in gold):", pred - gold)


## 12 — Optional: the same thing with NLTK

The slides reference **NLTK**'s parsers. If NLTK is installed in your Anaconda
environment, the cell below runs a real Viterbi (probabilistic CKY) parse and prints the
best tree — the library version of everything we hand-built in section 8. If NLTK isn't
installed it just prints a note; the notebook still stands on its own.


In [ ]:
try:
    import nltk
    from nltk import PCFG
    from nltk.parse import ViterbiParser

    g = PCFG.fromstring("""
        S  -> NP VP    [1.0]
        NP -> Det N    [1.0]
        VP -> V NP     [1.0]
        Det -> 'the'   [0.5]
        Det -> 'a'     [0.5]
        N  -> 'flight' [0.5]
        N  -> 'meal'   [0.5]
        V  -> 'includes' [1.0]
    """)
    parser = ViterbiParser(g)
    for tree in parser.parse("the flight includes a meal".split()):
        print("Best NLTK tree (prob = %.4g):" % tree.prob())
        print(tree)             # tree.pretty_print() draws it as ASCII art
except ImportError:
    print("NLTK not installed -> skipping. (conda install nltk)  The from-scratch")
    print("prob-CKY in section 8 already does exactly this, no dependencies needed.")


## Recap — one table to remember

| Concept | One-liner |
|---------|-----------|
| **Grammar** | formal spec of allowed structures; **CFG** = single symbol on the LHS |
| **Constituency vs Dependency** | nest words into phrases *vs* draw head→dependent arrows |
| **Top-down / Bottom-up** | search from `S`→words *vs* words→`S` |
| **Chart parsing** | cache partial results (`•` dot = active arc); no rework, no backtracking |
| **CKY** | bottom-up DP over spans; needs **CNF** (`A→B C` or `A→w`) |
| **PCFG** | every rule has a probability; per-LHS they sum to 1 |
| **P(tree)** | product of all rule probabilities; **best parse** = argmax |
| **Probabilistic CKY** | CKY carrying max-probabilities + back-pointers |
| **CNF** | binarise long rules, remove unit productions & mixed RHS |
| **PCFG weakness** | structural not lexical → no lexicalisation / context |
| **Evaluation** | precision / recall / **F1** on constituent brackets (PARSEVAL) |

**Next session (10):** *Dependency Parsing* — the other view of structure.
See [`../dependency-parsing/arc_eager_parser.ipynb`](../dependency-parsing/arc_eager_parser.ipynb).

---
*References (from the slide deck):* Jurafsky & Martin, *Speech and Language Processing*;
James Allen, *Natural Language Understanding*; Manning & Schütze, *Foundations of
Statistical NLP*; NLTK docs.
